# Exploration des fichiers sources :

*  Comparatif des colonnes
*  Harmonisation des types de colonnes
*  fusion pour obtenir deux fichiers comprenant les données de tous les fichiers source : incident_complet et mobilisations_complet
* conversion au format .parquet pour accélérer le traitement



### Import si usage Drive

In [1]:
from google.colab import drive
import os
drive.mount('/content/drive')
base_path = '/content/drive/MyDrive/Liora'

# Vérification si le dossier existe et affichage du contenu
if os.path.exists(base_path):
    print("Accès réussi au dossier Liora !")
    print("Contenu du dossier :", os.listdir(base_path))

    # Se déplacer dans ce dossier pour faciliter les imports suivants
    os.chdir(base_path)
else:
    print("Le dossier n'a pas été trouvé. Vérifie bien l'orthographe (majuscules/minuscules) ou si le dossier est dans un sous-dossier.")

Mounted at /content/drive
Accès réussi au dossier Liora !
Contenu du dossier : ['LFB Mobilisation data from January 2009 - 2014.xlsx', 'Mobilisations Metadata.xlsx', 'Incident Metadata.xlsx', 'LFB Incident data from 2024 onwards.xlsx', 'LFB Incident data from 2018 - 2023.xlsx', 'LFB Mobilisation data from 2015 - 2020.xlsx', 'LFB Mobilisation data from 2025.csv', 'LFB Mobilisation data from 2021 - 2024.csv', 'LFB Incident data from 2018 - 2023.csv', 'LFB Incident data from 2024 onwards.csv', 'LFB Mobilisation data from January 2009 - 2014.csv', 'LFB Mobilisation data from 2015 - 2020.csv', 'Metadata Incident_complet.docx', 'Metadata_Mobilisation_complet.docx', 'LFB Incident data from 2009 - 2017.csv', 'all_mobilisations.csv', 'mobilisation_completv26pm.csv', 'Incident_complet.csv', 'incident_complet.csv', 'mobilisation_complet.csv', 'incident_complet_2021_2025.parquet', 'mobilisation_complet_2021_2025.parquet', 'incident_complet.parquet', 'mobilisation_complet.parquet', 'incident_comple

### Import si usage local

In [2]:
base_path = ""

### Import Libraries

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Lecture des fichiers

In [4]:
#Fichiers Incidents
incident_1 = pd.read_csv('LFB Incident data from 2009 - 2017.csv',low_memory=False)
incident_2 = pd.read_excel('LFB Incident data from 2018 - 2023.xlsx')
incident_3 = pd.read_excel('LFB Incident data from 2024 onwards.xlsx')

In [5]:
# Fichiers Mobilisations
mobilisation_1 = pd.read_excel('LFB Mobilisation data from January 2009 - 2014.xlsx')
mobilisation_2 = pd.read_excel('LFB Mobilisation data from 2015 - 2020.xlsx')
mobilisation_3 = pd.read_csv('LFB Mobilisation data from 2021 - 2024.csv',low_memory=False)
mobilisation_4 = pd.read_csv('LFB Mobilisation data from 2025.csv',low_memory=False)

### Comparaison des fichiers : fonctions

In [6]:
def compare_columns_v2(dfs: dict, detail=True):
    cols_ref = set(next(iter(dfs.values())).columns)

    identique = True

    for nom, df in dfs.items():
        current_cols = set(df.columns)

        if current_cols != cols_ref:
            identique = False

            if detail:
                print(f"\n{nom} a des différences :")
                print("Colonnes en plus :", current_cols - cols_ref)
                print("Colonnes manquantes :", cols_ref - current_cols)

    return identique

In [7]:
def resume_df(df):
    resume = pd.DataFrame({
        "type": df.dtypes,
        "nb_valeurs_uniques": df.nunique(),
        "nb_valeurs_nulles": df.isna().sum(),
        "pourcentage_null": round((df.isna().sum() / len(df)) * 100, 2)
    })

    return resume.sort_values("nb_valeurs_uniques")

In [8]:
resume_df(mobilisation_1)

,type,nb_valeurs_uniques,nb_valeurs_nulles,pourcentage_null
DeployedFromLocation,object,2,44,0.00
PerformanceReporting,object,3,0,0.00
PlusCode_Code,object,3,0,0.00
PlusCode_Description,object,3,0,0.00
CalYear,int64,6,0,0.00
PumpOrder,int64,9,0,0.00
DelayCode_Description,object,10,673494,74.68
DelayCodeId,float64,10,673494,74.68
HourOfCall,int64,24,0,0.00
DeployedFromStation_Code,object,117,2,0.00


In [9]:
def compare_dtypes(dfs: dict):
    dtype_table = pd.DataFrame({k: v.dtypes.astype(str) for k, v in dfs.items()})
    diffs = dtype_table[dtype_table.nunique(axis=1) > 1].sort_index()
    print("Colonnes avec dtypes différents:", diffs.shape[0])
    return diffs

### Comparaison des fichiers sources, changement des types et concat : Incidents

In [10]:
# Liste des fichiers et colonnes à considérer pour la comparaison

dfsinc = {"inc1":incident_1,
          "inc2":incident_2,
          "inc3": incident_3}

{'colonnes_communes': {'AddressQualifier',
  'CalYear',
  'DateOfCall',
  'Easting_m',
  'Easting_rounded',
  'FRS',
  'FirstPumpArriving_AttendanceTime',
  'FirstPumpArriving_DeployedFromStation',
  'HourOfCall',
  'IncGeo_BoroughCode',
  'IncGeo_BoroughName',
  'IncGeo_WardCode',
  'IncGeo_WardName',
  'IncGeo_WardNameNew',
  'IncidentGroup',
  'IncidentNumber',
  'IncidentStationGround',
  'Latitude',
  'Longitude',
  'Northing_m',
  'Northing_rounded',
  'Notional Cost (£)',
  'NumCalls',
  'NumPumpsAttending',
  'NumStationsWithPumpsAttending',
  'Postcode_district',
  'Postcode_full',
  'ProperCase',
  'PropertyCategory',
  'PropertyType',
  'PumpCount',
  'PumpMinutesRounded',
  'SecondPumpArriving_AttendanceTime',
  'SecondPumpArriving_DeployedFromStation',
  'SpecialServiceType',
  'StopCodeDescription',
  'TimeOfCall',
  'UPRN',
  'USRN'},
 'colonnes differentes': {'inc1': {'colonnes_specifiques': set(),
   'colonnes_manquantes': set()},
  'inc2': {'colonnes_specifiques': set(), 'colonnes_manquantes': set()},
  'inc3': {'colonnes_specifiques': set(), 'colonnes_manquantes': set()}}}


{'colonnes_communes': {'AddressQualifier',
  'CalYear',
  'DateOfCall',
  'Easting_m',
  'Easting_rounded',
  'FRS',
  'FirstPumpArriving_AttendanceTime',
  'FirstPumpArriving_DeployedFromStation',
  'HourOfCall',
  'IncGeo_BoroughCode',
  'IncGeo_BoroughName',
  'IncGeo_WardCode',
  'IncGeo_WardName',
  'IncGeo_WardNameNew',
  'IncidentGroup',
  'IncidentNumber',
  'IncidentStationGround',
  'Latitude',
  'Longitude',
  'Northing_m',
  'Northing_rounded',
  'Notional Cost (£)',
  'NumCalls',
  'NumPumpsAttending',
  'NumStationsWithPumpsAttending',
  'Postcode_district',
  'Postcode_full',
  'ProperCase',
  'PropertyCategory',
  'PropertyType',
  'PumpCount',
  'PumpMinutesRounded',
  'SecondPumpArriving_AttendanceTime',
  'SecondPumpArriving_DeployedFromStation',
  'SpecialServiceType',
  'StopCodeDescription',
  'TimeOfCall',
  'UPRN',
  'USRN'},
 'colonnes differentes': {'inc1': {'colonnes_specifiques': set(),
   'colonnes_manquantes': set()},
  'inc2': {'colonnes_specifiques': set

In [11]:
compare_dtypes(dfsinc)

Colonnes avec dtypes différents: 3


,inc1,inc2,inc3
DateOfCall,object,datetime64[ns],datetime64[ns]
UPRN,float64,int64,int64
USRN,float64,int64,int64


In [12]:
#Conversion des types de colonnes différents
##incident_1['DateOfCall'] = pd.to_datetime(incident_1['DateOfCall'],format='%Y-%m-%d',errors='coerce')
incident_1['DateOfCall'] = pd.to_datetime(incident_1['DateOfCall'], errors='coerce', dayfirst=True)
incident_1['TimeOfCall'] = incident_1['TimeOfCall'].astype('object')
incident_1['UPRN'] = incident_1['UPRN'].astype('Int64')
incident_1['USRN'] = incident_1['USRN'].astype('Int64')
incident_2['UPRN'] = incident_2['UPRN'].astype('Int64')
incident_2['USRN'] = incident_2['USRN'].astype('Int64')
incident_3['UPRN'] = incident_3['UPRN'].astype('Int64')
incident_3['USRN'] = incident_3['USRN'].astype('Int64')

#Check du format DateofCall dans incident1
incident_1.info()

/tmp/ipykernel_622/133150662.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  incident_1['DateOfCall'] = pd.to_datetime(incident_1['DateOfCall'], errors='coerce', dayfirst=True)


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 988279 entries, 0 to 988278
Data columns (total 39 columns):
 #   Column                                  Non-Null Count   Dtype         
---  ------                                  --------------   -----         
 0   IncidentNumber                          988279 non-null  object        
 1   DateOfCall                              988279 non-null  datetime64[ns]
 2   CalYear                                 988279 non-null  int64         
 3   TimeOfCall                              988279 non-null  object        
 4   HourOfCall                              988279 non-null  int64         
 5   IncidentGroup                           988279 non-null  object        
 6   StopCodeDescription                     988279 non-null  object        
 7   SpecialServiceType                      299101 non-null  object        
 8   PropertyCategory                        988279 non-null  object        
 9   PropertyType                         

In [13]:
# Concat des fichiers pour obenir un seul dataframe
incident_complet = pd.concat([
    incident_1,
    incident_2,
    incident_3
], ignore_index=True
)

#check incident_complet
incident_complet.head()

,IncidentNumber,DateOfCall,CalYear,TimeOfCall,HourOfCall,IncidentGroup,StopCodeDescription,SpecialServiceType,PropertyCategory,PropertyType,...,FirstPumpArriving_AttendanceTime,FirstPumpArriving_DeployedFromStation,SecondPumpArriving_AttendanceTime,SecondPumpArriving_DeployedFromStation,NumStationsWithPumpsAttending,NumPumpsAttending,PumpCount,PumpMinutesRounded,Notional Cost (£),NumCalls
0,235138081.00,2009-01-01,2009,00:00:37,0,Special Service,Special Service,RTC,Road Vehicle,Car,...,319.0,Battersea,342.0,Clapham,2.0,2.0,2,60,255,1.0
1,1091.00,2009-01-01,2009,00:00:46,0,Special Service,Special Service,Assist other agencies,Outdoor,Lake/pond/reservoir,...,NaN,NaN,NaN,NaN,NaN,NaN,1,60,255,1.0
2,2091.00,2009-01-01,2009,00:03:00,0,Fire,Secondary Fire,NaN,Outdoor,Road surface/pavement,...,308.0,Edmonton,NaN,NaN,1.0,1.0,1,60,255,2.0
3,3091.00,2009-01-01,2009,00:04:27,0,Fire,Secondary Fire,NaN,Outdoor,Domestic garden (vegetation not equipment),...,210.0,Hillingdon,NaN,NaN,1.0,1.0,1,60,255,2.0
4,5091.00,2009-01-01,2009,00:05:39,0,Fire,Secondary Fire,NaN,Outdoor,Cycle path/public footpath/bridleway,...,233.0,Holloway,250.0,Holloway,1.0,2.0,2,60,255,1.0


In [14]:
# Conversion des types par pandas, suppression des duplicates et enregistrement
incident_complet.convert_dtypes()
incident_complet.drop_duplicates()
incident_complet.to_csv('incident_complet.csv',index=True)

#Check de incident_complet
incident_complet.head()

,IncidentNumber,DateOfCall,CalYear,TimeOfCall,HourOfCall,IncidentGroup,StopCodeDescription,SpecialServiceType,PropertyCategory,PropertyType,...,FirstPumpArriving_AttendanceTime,FirstPumpArriving_DeployedFromStation,SecondPumpArriving_AttendanceTime,SecondPumpArriving_DeployedFromStation,NumStationsWithPumpsAttending,NumPumpsAttending,PumpCount,PumpMinutesRounded,Notional Cost (£),NumCalls
0,235138081.00,2009-01-01,2009,00:00:37,0,Special Service,Special Service,RTC,Road Vehicle,Car,...,319.0,Battersea,342.0,Clapham,2.0,2.0,2,60,255,1.0
1,1091.00,2009-01-01,2009,00:00:46,0,Special Service,Special Service,Assist other agencies,Outdoor,Lake/pond/reservoir,...,NaN,NaN,NaN,NaN,NaN,NaN,1,60,255,1.0
2,2091.00,2009-01-01,2009,00:03:00,0,Fire,Secondary Fire,NaN,Outdoor,Road surface/pavement,...,308.0,Edmonton,NaN,NaN,1.0,1.0,1,60,255,2.0
3,3091.00,2009-01-01,2009,00:04:27,0,Fire,Secondary Fire,NaN,Outdoor,Domestic garden (vegetation not equipment),...,210.0,Hillingdon,NaN,NaN,1.0,1.0,1,60,255,2.0
4,5091.00,2009-01-01,2009,00:05:39,0,Fire,Secondary Fire,NaN,Outdoor,Cycle path/public footpath/bridleway,...,233.0,Holloway,250.0,Holloway,1.0,2.0,2,60,255,1.0


### Comparaison des fichiers sources, changement des types et concat : Mobilisations

In [15]:
#Liste des dataframes à considérer et comparatif des colonnes
dfsmob = {"mob1":mobilisation_1,
       "mob2":mobilisation_2,
       "mob3":mobilisation_3,
       "mob4":mobilisation_4}

compare_columns_v2(dfsmob)


mob3 a des différences :
Colonnes en plus : {'BoroughName', 'WardName'}
Colonnes manquantes : set()

mob4 a des différences :
Colonnes en plus : {'BoroughName', 'WardName'}
Colonnes manquantes : set()


False

In [16]:
#Conversion des types de colonnes différents

mobilisation_3['DateAndTimeArrived'] = pd.to_datetime(mobilisation_3['DateAndTimeArrived'],format="%d/%m/%Y %H:%M")
mobilisation_4['DateAndTimeArrived'] = pd.to_datetime(mobilisation_4['DateAndTimeArrived'],format="%d/%m/%Y %H:%M")

mobilisation_3['DateAndTimeLeft'] = pd.to_datetime(mobilisation_3['DateAndTimeLeft'],format="%d/%m/%Y %H:%M")
mobilisation_4['DateAndTimeLeft'] = pd.to_datetime(mobilisation_4['DateAndTimeLeft'],format="%d/%m/%Y %H:%M")

mobilisation_3['DateAndTimeMobile'] = pd.to_datetime(mobilisation_3['DateAndTimeMobile'],format="%d/%m/%Y %H:%M")
mobilisation_4['DateAndTimeMobile'] = pd.to_datetime(mobilisation_4['DateAndTimeMobile'],format="%d/%m/%Y %H:%M")

mobilisation_3['DateAndTimeMobilised'] = pd.to_datetime(mobilisation_3['DateAndTimeMobilised'],format="%d/%m/%Y %H:%M")
mobilisation_4['DateAndTimeMobilised'] = pd.to_datetime(mobilisation_4['DateAndTimeMobilised'],format="%d/%m/%Y %H:%M")

mobilisation_3['DateAndTimeReturned'] = pd.to_datetime(mobilisation_3['DateAndTimeReturned'],format="%d/%m/%Y %H:%M")
mobilisation_4['DateAndTimeReturned'] = pd.to_datetime(mobilisation_4['DateAndTimeReturned'],format="%d/%m/%Y %H:%M")

mobilisation_1['IncidentNumber'] = mobilisation_1['IncidentNumber'].astype('object')
mobilisation_3['IncidentNumber'] = mobilisation_3['IncidentNumber'].astype('object')
mobilisation_4['IncidentNumber'] = mobilisation_4['IncidentNumber'].astype('object')

mobilisation_3.head()

,IncidentNumber,CalYear,BoroughName,WardName,HourOfCall,ResourceMobilisationId,Resource_Code,PerformanceReporting,DateAndTimeMobilised,DateAndTimeMobile,...,DateAndTimeLeft,DateAndTimeReturned,DeployedFromStation_Code,DeployedFromStation_Name,DeployedFromLocation,PumpOrder,PlusCode_Code,PlusCode_Description,DelayCodeId,DelayCode_Description
0,000004-01012021,2021,HARINGEY,Muswell Hill,0,5769249,A321,1,2021-01-01 00:06:00,2021-01-01 00:07:00,...,2021-01-01 00:57:00,NaT,A32,Hornsey,Home Station,1,Initial,Initial Mobilisation,NaN,NaN
1,000005-01012021,2021,REDBRIDGE,MONKHAMS,0,5769250,F351,1,2021-01-01 00:07:00,2021-01-01 00:09:00,...,2021-01-01 00:18:00,NaT,F35,Woodford,Home Station,1,Initial,Initial Mobilisation,NaN,NaN
2,000006-01012021,2021,BARKING AND DAGENHAM,Village,0,5769251,F412,1,2021-01-01 00:08:00,2021-01-01 00:10:00,...,2021-01-01 00:24:00,NaT,F41,Dagenham,Home Station,1,Initial,Initial Mobilisation,12.0,Not held up
3,000007-01012021,2021,WANDSWORTH,West Hill,0,5769252,H331,1,2021-01-01 00:12:00,2021-01-01 00:13:00,...,2021-01-01 00:40:00,NaT,H33,Wandsworth,Home Station,1,Initial,Initial Mobilisation,8.0,Traffic calming measures
4,000007-01012021,2021,WANDSWORTH,West Hill,0,5769253,G351,2,2021-01-01 00:12:00,2021-01-01 00:13:00,...,2021-01-01 00:29:00,NaT,G35,Fulham,Home Station,2,Initial,Initial Mobilisation,NaN,NaN


In [17]:
#Fusion des fichiers
mobilisation_complet = pd.concat([
    mobilisation_1,
    mobilisation_2,
    mobilisation_3,
    mobilisation_4
], ignore_index=True
)

In [18]:
# Conversion des types par pandas, suppression des duplicates et enregistrement
mobilisation_complet = mobilisation_complet.convert_dtypes()
mobilisation_complet.drop_duplicates()
mobilisation_complet.to_csv('mobilisation_complet.csv',index=True)

#Check de mobilisation_complet
mobilisation_complet.head()

,IncidentNumber,CalYear,HourOfCall,ResourceMobilisationId,Resource_Code,PerformanceReporting,DateAndTimeMobilised,DateAndTimeMobile,DateAndTimeArrived,TurnoutTimeSeconds,...,DeployedFromStation_Code,DeployedFromStation_Name,DeployedFromLocation,PumpOrder,PlusCode_Code,PlusCode_Description,DelayCodeId,DelayCode_Description,BoroughName,WardName
0,235138081,2009,0,38426,H271,1,2009-01-01 00:02:27,NaT,2009-01-01 00:07:46,<NA>,...,H27,Battersea,Home Station,1,Initial,Initial Mobilisation,<NA>,<NA>,<NA>,<NA>
1,235138081,2009,0,38427,H212,2,2009-01-01 00:02:27,2009-01-01 00:06:40,2009-01-01 00:08:09,253,...,H21,Clapham,Home Station,2,Initial,Initial Mobilisation,<NA>,<NA>,<NA>,<NA>
2,2091,2009,0,38429,A341,1,2009-01-01 00:04:09,2009-01-01 00:06:40,2009-01-01 00:09:17,151,...,A34,Edmonton,Home Station,1,Initial,Initial Mobilisation,<NA>,<NA>,<NA>,<NA>
3,3091,2009,0,38430,G232,1,2009-01-01 00:04:57,2009-01-01 00:06:45,2009-01-01 00:08:27,108,...,G23,Hillingdon,Home Station,1,Initial,Initial Mobilisation,<NA>,<NA>,<NA>,<NA>
4,5091,2009,0,38432,A311,1,2009-01-01 00:06:04,2009-01-01 00:07:58,2009-01-01 00:09:57,114,...,A31,Holloway,Home Station,1,Initial,Initial Mobilisation,<NA>,<NA>,<NA>,<NA>


### Conversion au format parquet



In [19]:
#Instal Fastparquet
!pip install fastparquet

def preparer_pour_parquet(df):
    # On identifie les colonnes qui posent problème
    for col in df.columns:
        if df[col].dtype == 'object':
            # On force tout en string et on remplace les valeurs vides par une chaîne vide
            df[col] = df[col].astype(str).replace('None', '').replace('nan', '')
    return df

# Appliquer le nettoyage
incident_complet = preparer_pour_parquet(incident_complet)
mobilisation_complet = preparer_pour_parquet(mobilisation_complet)

# Sauvegarder
incident_complet.to_parquet('incident_complet.parquet', engine='fastparquet')
mobilisation_complet.to_parquet('mobilisation_complet.parquet', engine='fastparquet')

print("Fichiers sauvegardés avec succès !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 19.3 MB/s eta 0:00:00
Fichiers sauvegardés avec succès !


### Vérif des fichiers

In [20]:
import pandas as pd
incident_complet = pd.read_parquet('incident_complet.parquet', engine='fastparquet')
mobilisation_complet = pd.read_parquet('mobilisation_complet.parquet', engine='fastparquet')

display('Incident : ', incident_complet.shape, incident_complet.head())
print('__________________________________________________')
print('\n')
display('Mobilisations : ', mobilisation_complet.shape, mobilisation_complet.head())


'Incident : '

(1942988, 39)

,IncidentNumber,DateOfCall,CalYear,TimeOfCall,HourOfCall,IncidentGroup,StopCodeDescription,SpecialServiceType,PropertyCategory,PropertyType,...,FirstPumpArriving_AttendanceTime,FirstPumpArriving_DeployedFromStation,SecondPumpArriving_AttendanceTime,SecondPumpArriving_DeployedFromStation,NumStationsWithPumpsAttending,NumPumpsAttending,PumpCount,PumpMinutesRounded,Notional Cost (£),NumCalls
0,235138081.00,2009-01-01,2009,00:00:37,0,Special Service,Special Service,RTC,Road Vehicle,Car,...,319.0,Battersea,342.0,Clapham,2.0,2.0,2,60,255,1.0
1,1091.00,2009-01-01,2009,00:00:46,0,Special Service,Special Service,Assist other agencies,Outdoor,Lake/pond/reservoir,...,NaN,,NaN,,NaN,NaN,1,60,255,1.0
2,2091.00,2009-01-01,2009,00:03:00,0,Fire,Secondary Fire,,Outdoor,Road surface/pavement,...,308.0,Edmonton,NaN,,1.0,1.0,1,60,255,2.0
3,3091.00,2009-01-01,2009,00:04:27,0,Fire,Secondary Fire,,Outdoor,Domestic garden (vegetation not equipment),...,210.0,Hillingdon,NaN,,1.0,1.0,1,60,255,2.0
4,5091.00,2009-01-01,2009,00:05:39,0,Fire,Secondary Fire,,Outdoor,Cycle path/public footpath/bridleway,...,233.0,Holloway,250.0,Holloway,1.0,2.0,2,60,255,1.0


__________________________________________________




'Mobilisations : '

(2738795, 24)

,IncidentNumber,CalYear,HourOfCall,ResourceMobilisationId,Resource_Code,PerformanceReporting,DateAndTimeMobilised,DateAndTimeMobile,DateAndTimeArrived,TurnoutTimeSeconds,...,DeployedFromStation_Code,DeployedFromStation_Name,DeployedFromLocation,PumpOrder,PlusCode_Code,PlusCode_Description,DelayCodeId,DelayCode_Description,BoroughName,WardName
0,235138081,2009,0,38426,H271,1,2009-01-01 00:02:27,NaT,2009-01-01 00:07:46,<NA>,...,H27,Battersea,Home Station,1,Initial,Initial Mobilisation,<NA>,None,None,None
1,235138081,2009,0,38427,H212,2,2009-01-01 00:02:27,2009-01-01 00:06:40,2009-01-01 00:08:09,253,...,H21,Clapham,Home Station,2,Initial,Initial Mobilisation,<NA>,None,None,None
2,2091,2009,0,38429,A341,1,2009-01-01 00:04:09,2009-01-01 00:06:40,2009-01-01 00:09:17,151,...,A34,Edmonton,Home Station,1,Initial,Initial Mobilisation,<NA>,None,None,None
3,3091,2009,0,38430,G232,1,2009-01-01 00:04:57,2009-01-01 00:06:45,2009-01-01 00:08:27,108,...,G23,Hillingdon,Home Station,1,Initial,Initial Mobilisation,<NA>,None,None,None
4,5091,2009,0,38432,A311,1,2009-01-01 00:06:04,2009-01-01 00:07:58,2009-01-01 00:09:57,114,...,A31,Holloway,Home Station,1,Initial,Initial Mobilisation,<NA>,None,None,None


In [21]:
incident_complet.head()

,IncidentNumber,DateOfCall,CalYear,TimeOfCall,HourOfCall,IncidentGroup,StopCodeDescription,SpecialServiceType,PropertyCategory,PropertyType,...,FirstPumpArriving_AttendanceTime,FirstPumpArriving_DeployedFromStation,SecondPumpArriving_AttendanceTime,SecondPumpArriving_DeployedFromStation,NumStationsWithPumpsAttending,NumPumpsAttending,PumpCount,PumpMinutesRounded,Notional Cost (£),NumCalls
0,235138081.00,2009-01-01,2009,00:00:37,0,Special Service,Special Service,RTC,Road Vehicle,Car,...,319.0,Battersea,342.0,Clapham,2.0,2.0,2,60,255,1.0
1,1091.00,2009-01-01,2009,00:00:46,0,Special Service,Special Service,Assist other agencies,Outdoor,Lake/pond/reservoir,...,NaN,,NaN,,NaN,NaN,1,60,255,1.0
2,2091.00,2009-01-01,2009,00:03:00,0,Fire,Secondary Fire,,Outdoor,Road surface/pavement,...,308.0,Edmonton,NaN,,1.0,1.0,1,60,255,2.0
3,3091.00,2009-01-01,2009,00:04:27,0,Fire,Secondary Fire,,Outdoor,Domestic garden (vegetation not equipment),...,210.0,Hillingdon,NaN,,1.0,1.0,1,60,255,2.0
4,5091.00,2009-01-01,2009,00:05:39,0,Fire,Secondary Fire,,Outdoor,Cycle path/public footpath/bridleway,...,233.0,Holloway,250.0,Holloway,1.0,2.0,2,60,255,1.0
